In [0]:
# Imports

from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

## DIM Movies

In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_silver
        .withColumn("sk_movie_id", 
            F.row_number().over(Window.orderBy("id_filme")).cast("BIGINT"))
        .select(
            "sk_movie_id",
            F.col("id_filme").cast("string"),
            "titulo",
            "data_lancamento",
            "ano_lancamento",
            "duracao_minutos",
            "idioma_original",
            "status_filme",
            "sinopse"
        )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} linhas")

display(df_dim_movies.limit(5))

In [0]:
df_engajamento_silver = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_financeiro_silver = spark.table(f"{silver_schema}.tb_financeiro_filmes")

df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_fact_movies_performance = (
    df_engajamento_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .join(df_financeiro_silver, on="id_filme", how="inner")
        .select(
            "sk_movie_id",
            "orcamento_usd",
            "receita_usd",
            "lucro_usd",
            "orcamento_brl",
            "receita_brl",
            "lucro_brl",
            "popularidade",
            "nota_media_tmdb",
            "qtd_votos_tmdb",
            "nota_media_imdb",
            "qtd_votos_imdb"
        )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact_movies_performance.count()} linhas")

display(df_fact_movies_performance.limit(5))